In [ ]:
# Import required libraries for building a state graph workflow
# StateGraph: Core class for creating graph-based workflows
# START, END: Special nodes marking workflow entry and exit points
# TypedDict, Literal: Type hints for structured data and conditional routing
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [ ]:
# Define the state schema for the quadratic equation solver workflow
# This TypedDict specifies all data that flows through the workflow
class QuadState(TypedDict):
    # Input coefficients for ax^2 + bx + c = 0
    a: float
    b: float
    c: float

    # Computed values during workflow execution
    equation: str  # Formatted equation string
    discriminant: float  # b^2 - 4ac (determines root nature)
    result: str  # Final solution output

In [9]:
# Define all node functions for the quadratic equation workflow

def form_equation(state: QuadState):
    """Format the quadratic equation into a readable string."""
    a = state['a']
    b = state['b']
    c = state['c']

    if b<0:
        equation = f'{a}x^2 {b}x + {c}'
    else:
        equation = f'{a}x^2 + {b}x + {c}'
    return {'equation': equation}

def calculate_discriminant(state: QuadState):
    """Calculate the discriminant (b^2 - 4ac) to determine root type."""
    a = state['a']
    b = state['b']
    c = state['c']

    discriminant = b**2 - 4*a*c
    return {'discriminant': discriminant}

def two_distinct_real_roots(state: QuadState):
    """Calculate two distinct real roots when discriminant > 0."""
    root1 = (-state['b'] + state['discriminant']**0.5) / (2*state['a'])
    root2 = (-state['b'] - state['discriminant']**0.5) / (2*state['a'])

    result = f'Two distinct real roots: {root1} and {root2}'
    return {'result': result}

def one_repeated_real_root(state: QuadState):
    """Calculate one repeated real root when discriminant = 0."""
    root = -state['b'] / (2*state['a'])
    result = f'One repeated real root: {root}'
    return {'result': result}

def no_real_roots(state: QuadState):
    """Handle case with no real roots when discriminant < 0."""
    result = 'No real roots'
    return {'result': result}

def check_discriminant(state: QuadState) -> Literal['two_distinct_real_roots', 'one_repeated_real_root', 'no_real_roots']:
    """
    Conditional router that directs workflow based on discriminant value.
    Returns the appropriate next node name based on the discriminant.
    """
    if state['discriminant'] > 0:
        return "two_distinct_real_roots"
    elif state['discriminant'] == 0:
        return "one_repeated_real_root"
    else:
        return "no_real_roots"

In [10]:
# Build the quadratic equation solver workflow graph

# Create a state graph with QuadState as the data schema
graph = StateGraph(QuadState)

# Add all node functions to the graph
graph.add_node('form_equation', form_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)
graph.add_node('two_distinct_real_roots', two_distinct_real_roots)
graph.add_node('one_repeated_real_root', one_repeated_real_root)
graph.add_node('no_real_roots', no_real_roots)

# Define the workflow edges (execution path)
# Step 1: Start -> Form the equation string
graph.add_edge(START, 'form_equation')
# Step 2: Form equation -> Calculate discriminant
graph.add_edge('form_equation', 'calculate_discriminant')
# Step 3: Conditional routing based on discriminant value
graph.add_conditional_edges('calculate_discriminant', check_discriminant)
# Step 4: All root calculation paths lead to END
graph.add_edge('two_distinct_real_roots', END)
graph.add_edge('one_repeated_real_root', END)
graph.add_edge('no_real_roots', END)

# Compile the graph into an executable workflow
workflow = graph.compile()


In [ ]:
# Execute the workflow with an example quadratic equation
# Solving: 2x^2 + 4x + 2 = 0 (which has one repeated real root)
initial_state = {
    'a': 2,
    'b': 4,
    'c': 2
}

# Run the compiled workflow and display the complete state including results
workflow.invoke(initial_state)


{'a': 2,
 'b': 4,
 'c': 2,
 'equation': '2x^2 + 4x + 2',
 'discriminant': 0,
 'result': 'One repeated real root: -1.0'}